In [0]:
%pip install gspread google-auth
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

# Dynamically get last week's Sunday 00:00 -> this week's Sunday 00:00 (UK local time)
uk = ZoneInfo("Europe/London")
now_uk = datetime.now(uk)

# Python weekday(): Monday=0 ... Sunday=6
days_since_sunday = (now_uk.weekday() + 1) % 7
this_sunday = (now_uk - timedelta(days=days_since_sunday)).replace(hour=0, minute=0, second=0, microsecond=0)
last_sunday = this_sunday - timedelta(days=7)

from_date = last_sunday.strftime("%d/%m/%Y")
to_date   = this_sunday.strftime("%d/%m/%Y")

print(f"Automated Weekly Run -> Processing: {from_date} 00:00 to {to_date} 00:00 (UK local time)")

In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

uk = ZoneInfo("Europe/London")

start_local = last_sunday
end_local   = this_sunday

if end_local <= start_local:
    raise ValueError("To date must be after From date")

start_utc = start_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S")
end_utc   = end_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S")

date_time_range = f"{start_local.strftime('%d/%m/%Y %H:%M')} - {end_local.strftime('%d/%m/%Y %H:%M')}"

print(f"Week block: {date_time_range}")
print(f"UTC filter range: {start_utc} <= t < {end_utc}")

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"  # adjust to your path

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict, scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

# 5 destination sheets/tabs for MPF - Elmsall - Weekly
SHEET_TARGETS = {
    "PiE":          {"sheet_id": "1_w2Er6K9z5W_08313SLfXWs9bMItubo8d8Hvc3iXlu4", "gid": 424859743},
    "E3 Packing":   {"sheet_id": "1JKb63sTUC2oc56CSFB9GlrhyipGN-A7Xkd_Vhb77HnQ", "gid": 781522900},
    "Sorter 6":     {"sheet_id": "1HrfeWQm6F5uqHqmJ5wxOJvX3uLhzAMXWTOvtEFP7s2w", "gid": 671999882},
    "Parcel Sort":  {"sheet_id": "1pMRUkhGZ-ee5JVTp1abSh_Tbn7f_P2LrOZtb2sTYcvA", "gid": 1260169510},
    "E3 Top Up":    {"sheet_id": "1HfoJbvUhUxCMahELPDMTzqX7BqmT-2NYAVcA-rRln4w", "gid": 1429842033},
}

for name, target in SHEET_TARGETS.items():
    sh = gc.open_by_key(target["sheet_id"])
    ws = sh.get_worksheet_by_id(target["gid"])
    target["sh"] = sh
    target["ws"] = ws
    print(f"Connected [{name}]:", sh.title, "-> Tab:", ws.title)

In [0]:
# Read directly from the ABFSS delta path
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))
df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

# One config entry per destination sheet.
# Only "PiE" filter criteria have been confirmed so far -- the other 4 are left as
# None (PENDING) and will be skipped until their WAREHOUSECODE/AREACODE/EVENTTYPE
# and Area label are supplied.
all_reports = [
    {
        "name": "PiE",
        "warehouse_code": "X",
        "area_codes": ('Pick', 'Pie Station'),
        "event_types": ('COUN', 'MSKU', 'PSKU', 'RSIN'),
        "area_label": "MPF - PiE",
    },
    {
        "name": "E3 Packing",
        "warehouse_code": None,   # TODO: pending confirmation
        "area_codes": None,       # TODO: pending confirmation
        "event_types": None,      # TODO: pending confirmation
        "area_label": None,       # TODO: pending confirmation
    },
    {
        "name": "Sorter 6",
        "warehouse_code": None,   # TODO: pending confirmation
        "area_codes": None,       # TODO: pending confirmation
        "event_types": None,      # TODO: pending confirmation
        "area_label": None,       # TODO: pending confirmation
    },
    {
        "name": "Parcel Sort",
        "warehouse_code": None,   # TODO: pending confirmation
        "area_codes": None,       # TODO: pending confirmation
        "event_types": None,      # TODO: pending confirmation
        "area_label": None,       # TODO: pending confirmation
    },
    {
        "name": "E3 Top Up",
        "warehouse_code": None,   # TODO: pending confirmation
        "area_codes": None,       # TODO: pending confirmation
        "event_types": None,      # TODO: pending confirmation
        "area_label": None,       # TODO: pending confirmation
    },
]

# Exact columns matching the requested layout (including the 3 empty string placeholders)
COLUMNS = [
    'PAYLOAD_EVENTTYPE', 'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
    '', '', '',
    'Week', 'Date Time Range', 'Area'
]

print(f"Reports configured: {len(all_reports)}")
for r in all_reports:
    status = "READY" if r["warehouse_code"] else "PENDING config"
    print(f"  - {r['name']} [{status}]")

In [0]:
import pandas as pd

# Week number, matching the live job's formula, anchored to the start of this week block
week_val = spark.sql(f"""
  SELECT MOD(FLOOR(DATEDIFF(to_date(from_utc_timestamp(timestamp('{start_utc}'),'Europe/London')), DATE'2025-12-14')/7)+46,52) AS w
""").collect()[0]['w']

failures = []
total_rows_pushed = 0

print(f"\n=== Week: {date_time_range} ===")

for r in all_reports:
    name = r["name"]
    target = SHEET_TARGETS[name]
    ws = target["ws"]

    if not r["warehouse_code"] or not r["area_codes"] or not r["event_types"] or not r["area_label"]:
        print(f"  [{name}] SKIPPED -- filter criteria not yet configured")
        continue

    try:
        area_list = "(" + ", ".join(f"'{a}'" for a in r["area_codes"]) + ")"
        event_list = "(" + ", ".join(f"'{e}'" for e in r["event_types"]) + ")"

        query = f"""
          SELECT
            PAYLOAD_EVENTTYPE,
            SUM(PAYLOAD_QUANTITY)      AS Total_Quantity,
            SUM(PAYLOAD_STANDARDHOURS) AS Total_StandardHours,
            SUM(PAYLOAD_SMV)           AS Total_SMV
          FROM landing_bonus_hub_event_parsed
          WHERE TRIM(PAYLOAD_WAREHOUSECODE) = '{r["warehouse_code"]}'
            AND PAYLOAD_AREACODE IN {area_list}
            AND PAYLOAD_EVENTTYPE IN {event_list}
            AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{start_utc}')
            AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{end_utc}')
          GROUP BY 1
          ORDER BY PAYLOAD_EVENTTYPE
        """

        df_r = spark.sql(query).toPandas()

        df_r[''] = ''
        df_r['Week'] = week_val
        df_r['Date Time Range'] = date_time_range
        df_r['Area'] = r['area_label']

        df_r = df_r[COLUMNS]

        if len(df_r) == 0:
            df_r = pd.DataFrame(
                [[ '(no data this window)', '', '', '', '', '', '', week_val, date_time_range, r['area_label'] ]],
                columns=COLUMNS
            )

        values = df_r.astype(str).values.tolist()
        ws.append_rows(values, value_input_option='USER_ENTERED', table_range='A1')
        total_rows_pushed += len(df_r)
        print(f"  [{name}] {len(df_r)} rows -> {ws.title}")

    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        print(f"  [{name}] FAILED -- {error_msg}")
        failures.append({"report": name, "error": error_msg})
        continue

print(f"\n--- Weekly run summary ---")
print(f"Week processed: {date_time_range}")
print(f"Total rows pushed: {total_rows_pushed}")
print(f"Failures: {len(failures)}")
for f in failures:
    print(f"  {f['report']} | {f['error']}")